In [1]:
import gc
gc.collect()
%reset -f

In [2]:
# Install required packages
import numpy as np
import pandas as pd

In [3]:

from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from scipy.sparse import hstack,csr_matrix


In [ ]:
new_df=pd.read_csv(r'C:\nihal\python\NLP\QUORA_QA\Dataset\final.csv')
print(new_df.shape)
new_df = new_df.dropna(subset=['question1', 'question2', 'is_duplicate'])
new_df.drop("Unnamed: 0",axis=1,inplace=True)
new_df = new_df.reset_index(drop=True)
new_df.shape

(119964, 29)


(119955, 28)

In [5]:
results={}
new_df.sample()

,id,qid1,qid2,question1,question2,is_duplicate,q1_len,q2_len,q1_num_words,q2_num_words,...,ctc_max,last_word_eq,first_word_eq,abs_len_diff,mean_len,longest_substr_ratio,fuzz_ratio,fuzz_partial_ratio,token_sort_ratio,token_set_ratio
116196,116204,230350,230351,how can i get traffic on website,how do i increase organic traffic to website,1.0,32,44,7,8,...,0.499994,1.0,1.0,1.0,7.5,0.272727,68,70,66,79


In [6]:
q1 = list(new_df['question1'])
q2 = list(new_df['question2'])

In [7]:
new_df.drop(columns=['id','qid1','qid2','question1','question2'],inplace=True)
print(new_df.shape)
new_df.head()
x1=new_df.drop(['is_duplicate'],axis=1)
y=new_df['is_duplicate']
del new_df

(119955, 23)


In [8]:
# bag of words
from sklearn.feature_extraction.text import CountVectorizer
cv1 = CountVectorizer(max_features=3000)
cv1.fit(q1+q2)
q1_arr = cv1.transform(q1)   # sparse
q2_arr = cv1.transform(q2)   # sparse
# X = hstack([q1_arr, q2_arr])

In [9]:
cv1.vocabulary_

{'what': np.int64(2920),
 'is': np.int64(1437),
 'the': np.int64(2673),
 'step': np.int64(2541),
 'by': np.int64(416),
 'guide': np.int64(1217),
 'to': np.int64(2709),
 'invest': np.int64(1422),
 'in': np.int64(1364),
 'share': np.int64(2396),
 'market': np.int64(1650),
 'india': np.int64(1374),
 'story': np.int64(2552),
 'of': np.int64(1851),
 'diamond': np.int64(773),
 'how': np.int64(1315),
 'can': np.int64(435),
 'increase': np.int64(1370),
 'speed': np.int64(2506),
 'my': np.int64(1780),
 'internet': np.int64(1412),
 'connection': np.int64(607),
 'while': np.int64(2930),
 'using': np.int64(2821),
 'vpn': np.int64(2873),
 'why': np.int64(2936),
 'am': np.int64(148),
 'mentally': np.int64(1697),
 'very': np.int64(2842),
 'lonely': np.int64(1599),
 'solve': np.int64(2477),
 'it': np.int64(1444),
 'which': np.int64(2929),
 'one': np.int64(1870),
 'water': np.int64(2894),
 'sugar': np.int64(2587),
 'salt': np.int64(2309),
 'and': np.int64(162),
 'carbon': np.int64(449),
 'astrology': n

In [10]:

diff = np.abs(q1_arr - q2_arr)
print(q1_arr.shape)
print(q2_arr.shape)
mul = q1_arr.multiply(q2_arr)
print(diff.shape)
print(mul.shape)
print(x1.shape)
x_sparse= hstack([q1_arr, q2_arr, diff, mul])
x1_sparse = csr_matrix(x1)

# final combine
X = hstack([x1_sparse, x_sparse])



(119955, 3000)
(119955, 3000)
(119955, 3000)
(119955, 3000)
(119955, 22)


In [11]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [12]:

rf1= SGDClassifier()
rf1.fit(X_train,y_train)
y_pred = rf1.predict(X_test)
results['BoW + SGD']=accuracy_score(y_test,y_pred)


In [13]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

xgb1 = XGBClassifier(use_label_encoder=False, eval_metric='logloss',tree_method='hist')

xgb1.fit(X_train, y_train)

y_pred1 = xgb1.predict(X_test)
results['BoW + XGB']=accuracy_score(y_test,y_pred1) 
print(accuracy_score(y_test, y_pred1))

c:\nihal\python\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [21:37:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


0.8086782543453795


In [14]:
confusion_matrix(y_test,y_pred)

array([[11042,  3910],
       [ 1138,  7901]])

In [15]:
confusion_matrix(y_test,y_pred1)

array([[12746,  2206],
       [ 2384,  6655]])

In [16]:
del X_train, X_test, y_train, y_test


In [17]:
import gc
gc.collect()

31

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split

tfidf = TfidfVectorizer(max_features=5000)

# Fit
tfidf.fit(q1+q2)

# Transform (sparse)
q1_arr = tfidf.transform(q1)
q2_arr = tfidf.transform(q2)


In [19]:

diff = np.abs(q1_arr - q2_arr)
print(q1_arr.shape)
print(q2_arr.shape)
mul = q1_arr.multiply(q2_arr)
print(diff.shape)
print(mul.shape)
print(x1.shape)
x_sparse= hstack([q1_arr, q2_arr, diff, mul])
x1_sparse = csr_matrix(x1)

# final combine
X = hstack([x1_sparse, x_sparse])



(119955, 5000)
(119955, 5000)
(119955, 5000)
(119955, 5000)
(119955, 22)


In [20]:

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [21]:

rf2= SGDClassifier()
rf2.fit(X_train,y_train)
y_pred= rf2.predict(X_test)
results['Tfidf + SGD'] =accuracy_score(y_test,y_pred)


In [22]:
del q1_arr,q2_arr,X,diff,mul

In [23]:
from xgboost import XGBClassifier
xgb2 = XGBClassifier(tree_method='hist')
xgb2.fit(X_train,y_train)
y_pred1 = xgb2.predict(X_test)
results['Tfidf + XGB']=accuracy_score(y_test,y_pred1)


In [24]:
confusion_matrix(y_test,y_pred)

array([[14412,   540],
       [ 6082,  2957]])

In [25]:
confusion_matrix(y_test,y_pred1)

array([[12778,  2174],
       [ 2367,  6672]])

In [26]:
del X_train, X_test, y_train, y_test


In [27]:
!pip install -qqq bs4 python-Levenshtein fuzzywuzzy sentence_transformers nltk distance
!pip install gensim

import distance
from sentence_transformers import SentenceTransformer
from fuzzywuzzy import fuzz
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

c:\nihal\python\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [28]:
from nltk.stem import WordNetLemmatizer
import nltk

# download required data (run once)
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nihal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\nihal\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [29]:
def avg_word2vec(sentence, model, vector_size):
    words = sentence.split()
    vec = np.zeros(vector_size)
    count = 0

    for word in words:
        if word in model.wv:
            vec += model.wv[word]
            count += 1

    if count != 0:
        vec = vec / count

    return vec

In [30]:
#avg wrod2vec
tokenized_corpus = [sentence.split() for sentence in q1+q2]
model1= Word2Vec(
    sentences=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)
q1_arr = np.array([avg_word2vec(q, model1,100) for q in q1])
q2_arr = np.array([avg_word2vec(q, model1,100) for q in q2])



In [31]:
dif = np.abs(q1_arr - q2_arr)
mu = q1_arr * q2_arr
diff=csr_matrix(dif)
mul=csr_matrix(mu)
x_sparse = hstack([q1_arr, q2_arr, diff, mul])

# convert x1 to sparse
x1_sparse = csr_matrix(x1)

# final combine
X = hstack([x1_sparse, x_sparse])

print(X.shape)

(119955, 422)


In [32]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [33]:

rf3 = SGDClassifier()
rf3.fit(X_train,y_train)
y_pred = rf3.predict(X_test)
results['Avgword2vec + SGD'] =accuracy_score(y_test,y_pred)


In [34]:
del q1_arr,q2_arr,X

In [35]:
from xgboost import XGBClassifier
xgb3 = XGBClassifier(n_jobs=-1)
xgb3.fit(X_train,y_train)
y_pred1 = xgb3.predict(X_test)
results['Avgword2vec + XGB']=accuracy_score(y_test,y_pred1)


In [36]:
confusion_matrix(y_test,y_pred)

array([[5872, 9080],
       [ 205, 8834]])

In [37]:
confusion_matrix(y_test,y_pred1)

array([[12734,  2218],
       [ 2367,  6672]])

In [38]:
del X_train, X_test, y_train, y_test


In [39]:
def tfidf_word2vec(sentence, model, tfidf_dict, vector_size=100):
    words = sentence.split()

    vec = np.zeros(vector_size)
    weight_sum = 0

    for word in words:
        if word in model.wv and word in tfidf_dict:
            weight = tfidf_dict[word]
            vec += model.wv[word] * weight
            weight_sum += weight

    if weight_sum != 0:
        vec = vec / weight_sum

    return vec

In [40]:
#tfidf weighted word2vec
# as tfidf as previosly trained i will use that
# word → index mapping
tfidf_dict = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
q1_arr = np.array([
    tfidf_word2vec(q, model1, tfidf_dict, 100)
    for q in q1
])

q2_arr = np.array([
    tfidf_word2vec(q, model1, tfidf_dict, 100)
    for q in q2
])

In [41]:
dif = np.abs(q1_arr - q2_arr)
mu = q1_arr * q2_arr
diff=csr_matrix(dif)
mul=csr_matrix(mu)
x_sparse = hstack([q1_arr, q2_arr, diff, mul])

# convert x1 to sparse
x1_sparse = csr_matrix(x1)

# final combine
X = hstack([x1_sparse, x_sparse])

print(X.shape)

(119955, 422)


In [42]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [43]:

rf4= SGDClassifier()
rf4.fit(X_train,y_train)
y_pred = rf4.predict(X_test)
results['Tfidf +Avgword2vec + SGD']=accuracy_score(y_test,y_pred)


In [44]:
del X,q1_arr,q2_arr,x1_sparse,x_sparse,mul,mu,diff,dif

In [45]:
from xgboost import XGBClassifier
xgb4 = XGBClassifier(
    tree_method='hist',
    n_jobs=-1   
)
xgb4.fit(X_train,y_train)
y_pred1 = xgb4.predict(X_test)
results['Tfidf +Avgword2vec + XGB']=accuracy_score(y_test,y_pred1)


In [46]:
confusion_matrix(y_test,y_pred)

array([[ 3844, 11108],
       [   36,  9003]])

In [47]:
confusion_matrix(y_test,y_pred1)

array([[12701,  2251],
       [ 2318,  6721]])

In [48]:
del X_train, X_test, y_train, y_test


In [49]:
model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4122.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:

q1_arr = model.encode(q1)   
q2_arr = model.encode(q2) 


In [51]:
dif = np.abs(q1_arr - q2_arr)
mu = q1_arr * q2_arr
diff=csr_matrix(dif)
mul=csr_matrix(mu)
x_sparse = hstack([q1_arr, q2_arr, diff, mul])

# convert x1 to sparse
x1_sparse = csr_matrix(x1)

# final combine
X = hstack([x1_sparse, x_sparse])

print(X.shape)

(119955, 1558)


In [52]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [53]:

rf5=SGDClassifier()
rf5.fit(X_train,y_train)
y_pred = rf5.predict(X_test)
results['SBERT + SGD']=accuracy_score(y_test,y_pred)


In [54]:
del X,q1_arr,q2_arr,mu,mul,diff,dif,x1_sparse,x_sparse

In [55]:
from xgboost import XGBClassifier
xgb5=XGBClassifier(njobs=-1)
xgb5.fit(X_train,y_train)
y_pred1 = xgb5.predict(X_test)
results['SBERT + XGB']=accuracy_score(y_test,y_pred1)


c:\nihal\python\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [21:52:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "njobs" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [56]:
confusion_matrix(y_test,y_pred)

array([[11545,  3407],
       [ 1135,  7904]])

In [57]:
confusion_matrix(y_test,y_pred1)

array([[13050,  1902],
       [ 1792,  7247]])

In [58]:
del X_train, X_test, y_train, y_test


In [59]:
# print all scores and take better one
print(results)

{'BoW + SGD': 0.7895877620774457, 'BoW + XGB': 0.8086782543453795, 'Tfidf + SGD': 0.7239798257679964, 'Tfidf + XGB': 0.8107206869242632, 'Avgword2vec + SGD': 0.6129798674502939, 'Avgword2vec + XGB': 0.8088866658330207, 'Tfidf +Avgword2vec + SGD': 0.5354924763452962, 'Tfidf +Avgword2vec + XGB': 0.8095535825934725, 'SBERT + SGD': 0.810679004626735, 'SBERT + XGB': 0.8460255929306824}


In [60]:
#prediction

In [61]:
def test_common_words(q1,q2):
    w1 = set(map(lambda word: word.lower().strip(), q1.split(" ")))
    w2 = set(map(lambda word: word.lower().strip(), q2.split(" ")))
    return len(w1 & w2)
def test_total_words(q1,q2):
    w1 = set(map(lambda word: word.lower().strip(), q1.split(" ")))
    w2 = set(map(lambda word: word.lower().strip(), q2.split(" ")))
    return (len(w1) + len(w2))

In [62]:
def test_fetch_token_features(q1,q2):

    SAFE_DIV = 0.0001

    STOP_WORDS = stopwords.words("english")

    token_features = [0.0]*8

    q1_tokens = q1.split()
    q2_tokens = q2.split()

    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return token_features

    q1_words = set([word for word in q1_tokens if word not in STOP_WORDS])
    q2_words = set([word for word in q2_tokens if word not in STOP_WORDS])

    q1_stops = set([word for word in q1_tokens if word in STOP_WORDS])
    q2_stops = set([word for word in q2_tokens if word in STOP_WORDS])

    common_word_count = len(q1_words.intersection(q2_words))

    common_stop_count = len(q1_stops.intersection(q2_stops))

    common_token_count = len(set(q1_tokens).intersection(set(q2_tokens)))


    token_features[0] = common_word_count / (min(len(q1_words), len(q2_words)) + SAFE_DIV)
    token_features[1] = common_word_count / (max(len(q1_words), len(q2_words)) + SAFE_DIV)
    token_features[2] = common_stop_count / (min(len(q1_stops), len(q2_stops)) + SAFE_DIV)
    token_features[3] = common_stop_count / (max(len(q1_stops), len(q2_stops)) + SAFE_DIV)
    token_features[4] = common_token_count / (min(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)
    token_features[5] = common_token_count / (max(len(q1_tokens), len(q2_tokens)) + SAFE_DIV)

    token_features[6] = int(q1_tokens[-1] == q2_tokens[-1])

    token_features[7] = int(q1_tokens[0] == q2_tokens[0])

    return token_features


In [ ]:
def test_fetch_fuzzy_features(q1,q2):

    fuzzy_features = [0.0]*4

    fuzzy_features[0] = fuzz.QRatio(q1, q2)

    fuzzy_features[1] = fuzz.partial_ratio(q1, q2)

    fuzzy_features[2] = fuzz.token_sort_ratio(q1, q2)

    fuzzy_features[3] = fuzz.token_set_ratio(q1, q2)

    return fuzzy_features

In [64]:
import re
#by regular expressions
def remove_html_tags(txt):
    pat=re.compile('<.*?>')
    return pat.sub(r'',txt)
def remove_urls(text):
    pattern=re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'',text)
import string,time
ex=string.punctuation
def remove_punc(text):
    for char in ex:
        text=text.replace(char,'')
    return text

In [73]:
import re

nltk.download('stopwords')

STOP_WORDS = stopwords.words("english")
nltk.download('wordnet')


def preprocess(q):

    q = str(q).lower().strip()

    q = q.replace('%', ' percent')
    q = q.replace('$', ' dollar ')
    q = q.replace('₹', ' rupee ')
    q = q.replace('€', ' euro ')
    q = q.replace('@', ' at ')

    q = q.replace('[math]', '')

    q = q.replace(',000,000,000 ', 'b ')
    q = q.replace(',000,000 ', 'm ')
    q = q.replace(',000 ', 'k ')
    q = re.sub(r'([0-9]+)000000000', r'\1b', q)
    q = re.sub(r'([0-9]+)000000', r'\1m', q)
    q = re.sub(r'([0-9]+)000', r'\1k', q)

    con= {
    "ain't": "am not",
    "aren't": "are not",
    "can't": "can not",
    "can't've": "can not have",
    "'cause": "because",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'd've": "he would have",
    "he'll": "he will",
    "he'll've": "he will have",
    "he's": "he is",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how'll": "how will",
    "how's": "how is",
    "i'd": "i would",
    "i'd've": "i would have",
    "i'll": "i will",
    "i'll've": "i will have",
    "i'm": "i am",
    "i've": "i have",
    "isn't": "is not",
    "it'd": "it would",
    "it'd've": "it would have",
    "it'll": "it will",
    "it'll've": "it will have",
    "it's": "it is",
    "let's": "let us",
    "ma'am": "madam",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "o'clock": "of the clock",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shan't've": "shall not have",
    "she'd": "she would",
    "she'd've": "she would have",
    "she'll": "she will",
    "she'll've": "she will have",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so as",
    "that'd": "that would",
    "that'd've": "that would have",
    "that's": "that is",
    "there'd": "there would",
    "there'd've": "there would have",
    "there's": "there is",
    "they'd": "they would",
    "they'd've": "they would have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they're": "they are",
    "they've": "they have",
    "to've": "to have",
    "wasn't": "was not",
    "we'd": "we would",
    "we'd've": "we would have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what'll": "what will",
    "what'll've": "what will have",
    "what're": "what are",
    "what's": "what is",
    "what've": "what have",
    "when's": "when is",
    "when've": "when have",
    "where'd": "where did",
    "where's": "where is",
    "where've": "where have",
    "who'll": "who will",
    "who'll've": "who will have",
    "who's": "who is",
    "who've": "who have",
    "why's": "why is",
    "why've": "why have",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "you'd": "you would",
    "you'd've": "you would have",
    "you'll": "you will",
    "you'll've": "you will have",
    "you're": "you are",
    "you've": "you have"
    }

    q_de = []

    for word in q.split():
        if word in con:
            word = con[word]

        q_de.append(word)

    q = ' '.join(q_de)
    q = q.replace("'ve", " have")
    q = q.replace("n't", " not")
    q = q.replace("'re", " are")
    q = q.replace("'ll", " will")

    # Removing HTML tags
    q = remove_html_tags(q)

    # Remove punctuations
    q = remove_punc(q)
    #remove urls
    q = remove_urls(q)
    # lemmatization

    wordnet_lemmatizer = WordNetLemmatizer()
    q_le=[]
    for word in q.split():
        word = wordnet_lemmatizer.lemmatize(word)

        q_le.append(word)

    q = ' '.join(q_le)
    return q

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nihal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nihal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [74]:
def test_fetch_length_features(q1,q2):

    length_features = [0.0]*3

    q1_tokens = q1.split()
    q2_tokens = q2.split()

    if len(q1_tokens) == 0 or len(q2_tokens) == 0:
        return length_features

    length_features[0] = abs(len(q1_tokens) - len(q2_tokens))

    length_features[1] = (len(q1_tokens) + len(q2_tokens))/2

    strs = list(distance.lcsubstrings(q1, q2))
    length_features[2] = len(strs[0]) / (min(len(q1), len(q2)) + 1)

    return length_features

In [75]:
def query_point_creator(q1,q2,model2):

    input_query = []

    # preprocess
    q1 = preprocess(q1)
    q2 = preprocess(q2)

    # fetch basic features
    input_query.append(len(q1))
    input_query.append(len(q2))

    input_query.append(len(q1.split(" ")))
    input_query.append(len(q2.split(" ")))

    input_query.append(test_common_words(q1,q2))
    input_query.append(test_total_words(q1,q2))
    input_query.append(round(test_common_words(q1,q2)/test_total_words(q1,q2),2))

    token_features = test_fetch_token_features(q1,q2)
    input_query.extend(token_features)

    length_features = test_fetch_length_features(q1,q2)
    input_query.extend(length_features)

    fuzzy_features = test_fetch_fuzzy_features(q1,q2)
    input_query.extend(fuzzy_features)
    
    q1_vec =model2.encode([q1])
    q2_vec =model2.encode([q2])
    
    dif = np.abs(q1_vec - q2_vec)
    mu = q1_vec * q2_vec
    diff=csr_matrix(dif)
    mul=csr_matrix(mu)
    x_sparse = hstack([q1_vec, q2_vec, diff, mul])
    
    
    x1 = np.array(input_query).reshape(1, -1)
    x1_sparse = csr_matrix(x1)
    
    X_query = hstack([x1_sparse, x_sparse])
    return X_query



    

In [76]:
q1 = 'Where is the capital of India?'
q6 = 'Where is the capital of India?'
q2 = 'What is the current capital of Pakistan?'
q3 = 'Which city serves as the capital of India?'
q4 = 'What is the business capital of India?'
q5='hello'

In [72]:
xgb5.predict(query_point_creator(q1,q5,model))

array([0])

In [70]:
# cv1

In [ ]:
import pickle
import os

folder_path = r"C:\nihal\python\NLP\QUORA_QA\Models"

# create folder if not exists
os.makedirs(folder_path, exist_ok=True)

pickle.dump(xgb5, open(os.path.join(folder_path, "model.pkl"), "wb"))
pickle.dump(model, open(os.path.join(folder_path, "cv.pkl"), "wb"))